# Feature Engineering & Model Assessment

Exploratory analysis of the Day-Ahead vs. Imbalance spread and its relationship to fundamental grid features.
This notebook profiles the target variable used by the virtual trading strategy and examines which pre-auction features carry predictive signal.

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

proc = pd.read_parquet("../data/processed/processed_data.parquet")
proc["time"] = pd.to_datetime(proc["time"], utc=True)
proc = proc.sort_values("time").reset_index(drop=True)

proc["spread"] = proc["system_sell_price"] - proc["day_ahead_price"]

if "wind_fc_da_d1_10h30" in proc.columns and "demand_fc_da_d1_10h30" in proc.columns:
    proc["auction_residual_load"] = proc["demand_fc_da_d1_10h30"] - proc["wind_fc_da_d1_10h30"]

print(f"Dataset: {proc['time'].min().date()} to {proc['time'].max().date()}")
print(f"Half-hour periods: {len(proc):,}")
print(f"Columns: {len(proc.columns)}")

---
## Section 1 — The Target Variable

The virtual strategy profits from the spread between the Day-Ahead auction price and the imbalance settlement price (SSP):

$$\text{Spread} = \text{SSP} - \text{DA Price}$$

Understanding the statistical properties of this spread is essential: heavy tails and skewness determine whether mean-based models are appropriate, or whether the strategy must account for extreme deviations that dominate P&L.

In [ ]:
spread = proc["spread"].dropna()

skewness = stats.skew(spread)
kurt = stats.kurtosis(spread)

print("=== Spread Distribution: SSP − DA Price (GBP/MWh) ===\n")
print(f"  Count:       {len(spread):,}")
print(f"  Mean:        {spread.mean():+.2f}")
print(f"  Median:      {spread.median():+.2f}")
print(f"  Std Dev:     {spread.std():.2f}")
print(f"  Min:         {spread.min():+.1f}")
print(f"  Max:         {spread.max():+.1f}")
print(f"  Skewness:    {skewness:+.3f}")
print(f"  Ex. Kurtosis:{kurt:+.3f}")
print()
if abs(kurt) > 3:
    print(f"  Kurtosis of {kurt:+.1f} confirms heavy tails — extreme spread events")
    print("  occur far more often than a normal distribution would predict.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(spread, bins=120, density=True, alpha=0.6, color="#1565C0", edgecolor="none")
spread.plot.kde(ax=axes[0], linewidth=2, color="#B71C1C", label="KDE")
axes[0].axvline(spread.mean(), color="#E65100", linestyle="--", linewidth=1.2, label=f"Mean ({spread.mean():+.1f})")
axes[0].axvline(spread.median(), color="#2E7D32", linestyle="--", linewidth=1.2, label=f"Median ({spread.median():+.1f})")
axes[0].set_xlabel("Spread: SSP − DA Price (GBP/MWh)", fontsize=11)
axes[0].set_ylabel("Density", fontsize=11)
axes[0].set_title("Distribution of Imbalance Spread", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].set_xlim(-150, 200)

axes[1].boxplot(spread, vert=True, widths=0.5,
                boxprops=dict(color="#1565C0"),
                medianprops=dict(color="#B71C1C", linewidth=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))
axes[1].set_ylabel("Spread (GBP/MWh)", fontsize=11)
axes[1].set_title("Spread Box Plot — Outlier Structure", fontsize=12)
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

---
## Section 2 — Feature Correlation

We examine three core grid fundamentals available before the 11:00 AM EPEX auction and their relationship to the target spread:

| Feature | Description |
|---|---|
| **`auction_residual_load`** | Demand forecast minus wind forecast at the D-1 10:30 AM snapshot. Captures the net thermal generation requirement — the primary driver of DA price formation. |
| **`wind_fc_da_d1_10h30`** | Wind generation forecast at the auction snapshot. Renewable intermittency directly impacts both DA clearing price and real-time imbalance. |
| **`demand_fc_da_d1_10h30`** | Demand forecast at the auction snapshot. Defines the total system requirement that must be met — sets the baseline for residual load. |

In [ ]:
features = ["auction_residual_load", "wind_fc_da_d1_10h30", "demand_fc_da_d1_10h30"]
labels = ["Auction Residual Load (MW)", "Wind Forecast — D-1 10:30 (MW)", "Demand Forecast — D-1 10:30 (MW)"]

scatter_df = proc[["spread"] + features].dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feat, label in zip(axes, features, labels):
    ax.scatter(scatter_df[feat], scatter_df["spread"],
               alpha=0.15, s=8, color="#1565C0", edgecolors="none")
    r = scatter_df[feat].corr(scatter_df["spread"])
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel("Spread: SSP − DA (GBP/MWh)", fontsize=10)
    ax.set_title(f"r = {r:+.3f}", fontsize=11)
    ax.axhline(0, color="#9E9E9E", linewidth=0.8, linestyle="--")

fig.suptitle("Feature vs. Imbalance Spread — Scatter Plots", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
heatmap_cols = [
    "spread",
    "auction_residual_load",
    "wind_fc_da_d1_10h30",
    "demand_fc_da_d1_10h30",
    "day_ahead_price",
    "system_sell_price",
]
heatmap_labels = [
    "Spread (SSP−DA)",
    "Auction Residual Load",
    "Wind Forecast (D-1)",
    "Demand Forecast (D-1)",
    "DA Price",
    "SSP",
]

corr = proc[heatmap_cols].dropna().corr()
corr.index = heatmap_labels
corr.columns = heatmap_labels

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    square=True,
    cbar_kws={"label": "Pearson Correlation", "shrink": 0.8},
)
ax.set_title("Correlation Heatmap — Spread & Auction Fundamentals", fontsize=13, pad=12)
plt.tight_layout()
plt.show()